# Chopp & Cia · 03 — Análise Exploratória

**Projeto Integrador VI** · FATEC Votorantim · 2º Semestre/2026

Este notebook responde, em linguagem de negócio, a uma pergunta:

> **Quem são os clientes que dão prejuízo, e o que eles têm em comum?**

A empresa cede chopeiras em comodato e vende chopp a prazo. Isso cria dois riscos
que caminham juntos: o cliente **não paga** as parcelas, e o cliente **não devolve**
o equipamento. As análises abaixo medem os dois — e mostram, com dado, onde a
cobrança rende mais por esforço.

| | |
|:---|:---|
| **Entrada** | consolidado do notebook 01 (CSV) ou a tabela do 02 |
| **Saída** | leitura de negócio — este notebook **não altera dado nenhum** |
| **Ambiente** | Windows local ou Databricks |

---

### Como ler este notebook

Cada seção tem três partes, sempre na mesma ordem:

1. **A pergunta** — o que se quer saber.
2. **O gráfico** — a resposta visual.
3. **A leitura** — o que o número significa para a operação, impresso abaixo do
   gráfico.

Você não precisa ler o código. As conclusões estão no texto e nos gráficos.

### Roteiro

| § | Pergunta |
|:-:|:---|
| 3 | Com quantos clientes a empresa trabalha? |
| 4 | Quanto da carteira está em risco, e quanto dinheiro isso representa? |
| 5 | Há quanto tempo os atrasos se arrastam? |
| 6 | O risco tem endereço, forma de pagamento, tipo de negócio? |
| 7 | Cliente que sumiu é cliente que não pagou? |
| 8 | Comprar a prazo prevê inadimplência? |
| 9 | Onde está o dinheiro — e o equipamento — em jogo? |
| 10 | Quantos clientes o pipeline descarta, e por quê? |
| 11 | Onde colocar a régua do atraso? |
| 12 | Quantas compras bastam para o número significar algo? |
| 13 | Alternar entre à vista e a prazo é sinal de risco? |
| 14 | O que foi deliberadamente deixado fora do modelo? |
| 15 | Resumo e o que fica para os próximos notebooks |

As seções 10 a 14 são o **estudo de parâmetros**: elas não descrevem a carteira,
elas examinam as decisões que o pipeline toma sobre ela — o recorte da população,
a régua do atraso, o histórico mínimo e as regras de governança.

> **Sobre privacidade:** nenhum cliente é identificado por nome. A análise usa
> apenas o código do cliente (`ID_PESSOA`). Para saber de quem se trata,
> consulte o ERP com esse código.

## 1. De onde vêm os dados

Preencha o caminho conforme onde estiver rodando. No Databricks, informe a tabela
publicada pelo notebook 02; localmente, o CSV que o notebook 01 gerou.

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

CAMINHO_CSV = r""
TABELA = "projetointegrador.projetointegrador.dataset_consolidado_v1_0" # Databricks

try:
    spark                                                      # noqa: F821
    EM_DATABRICKS = True
except NameError:
    EM_DATABRICKS = False

if EM_DATABRICKS:
    dados = spark.table(TABELA).toPandas()                     # noqa: F821
    origem = TABELA
else:
    from pathlib import Path
    if not CAMINHO_CSV.strip():
        raise ValueError("Preencha CAMINHO_CSV com o CSV gerado pelo notebook 01.")
    caminho = Path(CAMINHO_CSV.strip())
    if caminho.is_dir():
        raise IsADirectoryError(f"CAMINHO_CSV aponta para uma pasta, nao para o arquivo .csv: {caminho}")
    if not caminho.exists():
        raise FileNotFoundError(f"CSV não encontrado: {caminho}")
    dados = pd.read_csv(caminho, sep=";", encoding="utf-8-sig", low_memory=False)
    origem = caminho.name

dados.columns = [str(c).strip().upper() for c in dados.columns]

NUMERICAS = [
    "DIAS_DESDE_PRIMEIRA_COMPRA", "DIAS_DESDE_ULTIMA_COMPRA", "FREQUENCIA_COMPRAS",
    "TOTAL_ITENS", "QTD_TOTAL_VENDIDA", "TOTAL_GASTO", "TICKET_MEDIO", "PCT_COMPRAS_A_PRAZO",
    "TOTAL_PARCELAS", "PARCELAS_ATRASADAS", "TAXA_ATRASO_PAGAMENTO",
    "MEDIA_DIAS_ATRASO_PAG", "MAX_DIAS_ATRASO_PAG", "VALOR_TOTAL_PARCELAS",
    "TOTAL_COMODATOS", "COMODATOS_ATRASADOS", "TAXA_ATRASO_COMODATO",
    "MEDIA_DIAS_ATRASO_COM", "MAX_DIAS_ATRASO_COM", "QTD_EQUIPAMENTOS",
    "PRAZO_MEDIO_COMODATO", "VALOR_MEDIO_COMODATO",
    "RISCO_FINANCEIRO", "RISCO_COMODATO", "TEM_COMISSAO",
    "CORE_BUSINESS", "TEM_VENDAS", "TEM_FINANCEIRO", "TEM_COMODATO",
]
for coluna in NUMERICAS:
    dados[coluna] = pd.to_numeric(dados[coluna], errors="coerce")
for coluna in ["PRIMEIRA_COMPRA", "ULTIMA_COMPRA"]:
    dados[coluna] = pd.to_datetime(dados[coluna], errors="coerce")

# Régua de risco usada pelas seções analíticas (11 a 14). É a MESMA que o
# notebook 01 aplicou para gravar RISCO_FINANCEIRO/RISCO_COMODATO — declarada
# aqui porque estas seções variam o valor para medir sensibilidade.
LIMITE_RISCO = 0.20

print(f"Origem   : {origem}")
print(f"Carteira : {len(dados):,} clientes")

## 2. Como os gráficos são desenhados

As cores não são escolha de gosto. Elas seguem três regras que garantem que o
gráfico continue legível para quem tem daltonismo e quando impresso:

- **Severidade** (aging) usa **um tom só de azul**, do claro ao escuro — mais
  escuro é pior. A ordem das barras já carrega a informação; a cor reforça.
- **Categorias** (perfil de risco) usam cores separadas e testadas para não se
  confundirem sob daltonismo — e toda fatia leva rótulo escrito, nunca só cor.
- **Comparações de duas trilhas** (financeiro × comodato) usam azul e laranja,
  o par de maior separação visual.

In [0]:
COR_TEXTO, COR_APOIO, COR_GRADE = "#0b0b0b", "#52514e", "#e3e2df"
COR_NEUTRA = "#b8b7b2"

# Rampa sequencial de um tom só: mais escuro = mais grave.
RAMPA_SEVERIDADE = ["#86b6ef", "#3987e5", "#256abf", "#184f95", "#0d366b"]

# Categóricas separadas sob protanopia e deuteranopia (verificado antes de usar).
CORES_PERFIL_RISCO = {
    "SEM RISCO": "#008300",
    "SÓ COMODATO": "#eda100",
    "SÓ FINANCEIRO": "#e87ba4",
    "RISCO DUPLO": "#4a3aa7",
}
COR_FINANCEIRO, COR_COMODATO = "#2a78d6", "#eb6834"

ORDEM_AGING = ["Sem Atraso", "1-3 Dias", "4-7 Dias", "8-15 Dias",
               "16-20 Dias", "21-30 Dias", "+30 Dias"]
ORDEM_PERFIL_RISCO = ["SEM RISCO", "SÓ COMODATO", "SÓ FINANCEIRO", "RISCO DUPLO"]

plt.rcParams.update({
    "figure.dpi": 110, "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.edgecolor": COR_GRADE, "axes.labelcolor": COR_APOIO,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlecolor": COR_TEXTO,
    "axes.grid": True, "grid.color": COR_GRADE, "grid.linewidth": 0.8,
    "axes.axisbelow": True,          # grade ATRAS das barras
    "xtick.color": COR_APOIO, "ytick.color": COR_APOIO,
    "font.size": 10, "axes.spines.top": False, "axes.spines.right": False,
})


def cor_severidade(i, n):
    """Escolhe o degrau da rampa proporcional à posição na escala de severidade."""
    if n <= 1:
        return RAMPA_SEVERIDADE[-1]
    passo = i * (len(RAMPA_SEVERIDADE) - 1) / (n - 1)
    return RAMPA_SEVERIDADE[int(round(passo))]


def moeda(valor, _=None):
    if valor >= 1e6:
        return f"R$ {valor / 1e6:.1f}M"
    if valor >= 1e3:
        return f"R$ {valor / 1e3:.0f}k"
    return f"R$ {valor:.0f}"


def milhar(valor, _=None):
    return f"{int(valor):,}".replace(",", ".")


def leitura(titulo, linhas):
    """Imprime a interpretação de negócio abaixo do gráfico."""
    print(f"\n{titulo}")
    print("-" * max(len(titulo), 40))
    for linha in linhas:
        print(f"  {linha}")


def rotular_barras(ax, barras, valores, formato=milhar, folga=0.01):
    """Escreve o valor ao lado de cada barra — identidade nunca fica só na cor."""
    limite = ax.get_xlim()[1]
    for barra, valor in zip(barras, valores):
        ax.text(barra.get_width() + limite * folga,
                barra.get_y() + barra.get_height() / 2,
                formato(valor), va="center", fontsize=9, color=COR_TEXTO)


print("Padrão visual carregado.")

## Visão geral em quatro números

Antes de entrar seção por seção, o retrato da carteira hoje.

In [0]:
_em_risco = int((dados["PERFIL_RISCO"] != "SEM RISCO").sum())
_exposto = float(dados.loc[dados["PERFIL_RISCO"] != "SEM RISCO", "TOTAL_GASTO"].sum())

KPIS = [
    ("clientes na carteira", milhar(total := len(dados)), COR_NEUTRA),
    ("em algum tipo de risco", f"{_em_risco / total * 100:.0f}%", RAMPA_SEVERIDADE[2]),
    ("faturamento exposto", moeda(_exposto), RAMPA_SEVERIDADE[3]),
    ("em risco duplo (o mais crítico)",
     milhar(int((dados['PERFIL_RISCO'] == 'RISCO DUPLO').sum())), CORES_PERFIL_RISCO["RISCO DUPLO"]),
]

fig, eixos = plt.subplots(1, 4, figsize=(13, 2.3))
for ax, (rotulo, valor, cor) in zip(eixos, KPIS):
    ax.set_axis_off()
    ax.text(0.5, 0.62, valor, ha="center", va="center", fontsize=26, fontweight="bold",
            color=cor, transform=ax.transAxes)
    ax.text(0.5, 0.18, rotulo, ha="center", va="center", fontsize=10.5, color=COR_APOIO,
            transform=ax.transAxes, wrap=True)
plt.tight_layout()
plt.show()

## 3. O tamanho da carteira

Antes de falar de risco, é preciso saber de quantos clientes se está falando — e
quantos deles a empresa realmente conhece.

Nem todo cliente cadastrado comprou; nem todo comprador pegou equipamento em
comodato. Um cliente sem histórico não é um cliente de baixo risco: é um cliente
**sobre o qual não se sabe nada**, e a diferença importa na hora de conceder
crédito.

In [0]:
# CORREÇÃO APLICADA: A flag TEM_VENDAS original estava inconsistente — havia clientes
# com parcelas ou equipamentos MAS SEM vendas, o que é logicamente impossível.
# A correção abaixo garante que TEM_VENDAS inclua QUALQUER cliente com transação.

# Corrigir TEM_VENDAS: incluir clientes com financeiro OU comodato
dados['TEM_VENDAS_CORRIGIDO'] = (
    (dados['TEM_VENDAS'] == 1) |
    (dados['TEM_FINANCEIRO'] == 1) |
    (dados['TEM_COMODATO'] == 1)
).astype(int)

# Contagens corrigidas
total = len(dados)
com_transacao = int(dados['TEM_VENDAS_CORRIGIDO'].sum())
com_parcelas = int(dados['TEM_FINANCEIRO'].sum())
com_equipamento = int(dados['TEM_COMODATO'].sum())
core_business = int(dados['CORE_BUSINESS'].sum())

# Criar 5 categorias MUTUAMENTE EXCLUSIVAS
apenas_cadastrado = (
    (dados['TEM_VENDAS_CORRIGIDO'] == 0) & 
    (dados['TEM_FINANCEIRO'] == 0) & 
    (dados['TEM_COMODATO'] == 0)
)
a_vista_puro = (
    (dados['TEM_VENDAS_CORRIGIDO'] == 1) & 
    (dados['TEM_FINANCEIRO'] == 0) & 
    (dados['TEM_COMODATO'] == 0)
)
so_financeiro = (
    (dados['TEM_FINANCEIRO'] == 1) & 
    (dados['TEM_COMODATO'] == 0)
)
so_comodato = (
    (dados['TEM_FINANCEIRO'] == 0) & 
    (dados['TEM_COMODATO'] == 1)
)
financeiro_comodato = (
    (dados['TEM_FINANCEIRO'] == 1) & 
    (dados['TEM_COMODATO'] == 1)
)

# Criar visualização em dois gráficos: alcance + composição exclusiva (VERTICAL)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

# ========================================
# GRÁFICO 1: ALCANCE/COBERTURA DO NEGÓCIO
# ========================================
categorias_alcance = [
    'Clientes\ncadastrados',
    'Com alguma\ntransação',
    'Com parcelas\na receber',
    'Com equipamento\ncedido',
    'Compraram\nchopp/chopeira'
]
valores_alcance = [total, com_transacao, com_parcelas, com_equipamento, core_business]
cores_alcance = [COR_NEUTRA, '#3987e5', '#256abf', '#eb6834', '#7851a9']

barras1 = ax1.barh(categorias_alcance[::-1], valores_alcance[::-1], 
                   color=cores_alcance[::-1], height=0.58)
ax1.xaxis.set_major_formatter(FuncFormatter(milhar))
ax1.set_xlim(0, total * 1.25)
ax1.set_title("Alcance e Cobertura do Negócio", fontweight='bold', fontsize=11.5)
ax1.set_xlabel("número de clientes")
ax1.grid(axis="y", visible=False)

for barra, valor in zip(barras1, valores_alcance[::-1]):
    pct = (valor / total * 100)
    label = f"{milhar(valor)}  ({pct:.0f}%)"
    ax1.text(barra.get_width() + total * 0.02, 
             barra.get_y() + barra.get_height() / 2,
             label, va="center", fontsize=9.5, color=COR_TEXTO, fontweight='bold')

# ========================================
# GRÁFICO 2: COMPOSIÇÃO POR CATEGORIA EXCLUSIVA (BARRAS HORIZONTAIS INDIVIDUAIS)
# ========================================
cores_categorias = {
    'Financeiro + Comodato': '#7851a9',
    'Só Financeiro (A Prazo)': '#2a78d6',
    'Só Comodato': '#eb6834',
    'Comprador À Vista Puro': '#4caf50',
    'Apenas Cadastrado (Inativo)': COR_NEUTRA
}

# Preparar dados ordenados por volume decrescente
categorias_dados = [
    ('Financeiro + Comodato', int(financeiro_comodato.sum())),
    ('Só Financeiro (A Prazo)', int(so_financeiro.sum())),
    ('Só Comodato', int(so_comodato.sum())),
    ('Comprador À Vista Puro', int(a_vista_puro.sum())),
    ('Apenas Cadastrado (Inativo)', int(apenas_cadastrado.sum()))
]
# Ordenar por contagem decrescente
categorias_dados_sorted = sorted(categorias_dados, key=lambda x: x[1], reverse=True)
categorias_sorted = [cat for cat, _ in categorias_dados_sorted]
contagens_sorted = [count for _, count in categorias_dados_sorted]
cores_sorted = [cores_categorias[cat] for cat in categorias_sorted]

# Criar barras horizontais individuais
barras2 = ax2.barh(categorias_sorted, contagens_sorted, color=cores_sorted, height=0.62)

ax2.xaxis.set_major_formatter(FuncFormatter(milhar))
ax2.set_xlim(0, max(contagens_sorted) * 1.25)  # Margem para rótulos externos
ax2.set_title("Composição da Carteira por Categoria", fontweight='bold', fontsize=11.5)
ax2.set_xlabel("número de clientes")
ax2.grid(axis="y", visible=False)

# Adicionar rótulos FORA das barras (externa) para todas as categorias
for barra, count in zip(barras2, contagens_sorted):
    pct = (count / total * 100)
    label = f"{milhar(count)}  ({pct:.0f}%)"
    ax2.text(barra.get_width() + max(contagens_sorted) * 0.02,
             barra.get_y() + barra.get_height() / 2,
             label, va='center', fontsize=9.5, color=COR_TEXTO, fontweight='bold')

plt.tight_layout()
plt.show()

leitura("O que isso significa", [
    f"A carteira tem {total:,} clientes cadastrados.",
    "",
    "COMPOSIÇÃO DA CARTEIRA (5 categorias mutuamente exclusivas):",
    f"  • Financeiro + Comodato:       {int(financeiro_comodato.sum()):,} ({int(financeiro_comodato.sum())/total*100:.0f}%) — RISCO DUPLO",
    f"  • Só Financeiro (A Prazo):     {int(so_financeiro.sum()):,} ({int(so_financeiro.sum())/total*100:.0f}%)",
    f"  • Só Comodato:                 {int(so_comodato.sum()):,} ({int(so_comodato.sum())/total*100:.0f}%)",
    f"  • Comprador À Vista Puro:      {int(a_vista_puro.sum()):,} ({int(a_vista_puro.sum())/total*100:.0f}%) — ativos sem exposição",
    f"  • Apenas Cadastrado (Inativo): {int(apenas_cadastrado.sum()):,} ({int(apenas_cadastrado.sum())/total*100:.0f}%) — sem histórico",
    "",
    f"TOTAL ATIVOS:  {int(a_vista_puro.sum() + so_financeiro.sum() + so_comodato.sum() + financeiro_comodato.sum()):,} "
    f"({(a_vista_puro.sum() + so_financeiro.sum() + so_comodato.sum() + financeiro_comodato.sum())/total*100:.0f}%)",
    f"TOTAL INATIVOS: {int(apenas_cadastrado.sum()):,} ({int(apenas_cadastrado.sum())/total*100:.0f}%)",
    "",
    f"O grupo RISCO DUPLO ({int(financeiro_comodato.sum()):,} clientes, {int(financeiro_comodato.sum())/total*100:.0f}%) é prioritário:",
    "devem dinheiro E estão com equipamento — máxima exposição e poder de negociação.",
    "",
    f"Os {int(apenas_cadastrado.sum())} clientes inativos não têm histórico: não são de baixo risco,",
    "são desconhecidos. Conceder crédito a eles é decisão sem base em dado.",
])

## 4. Quanto da carteira está em risco

Um cliente entra em risco quando **mais de 20% dos seus compromissos atrasam** —
seja no pagamento das parcelas, seja na devolução do equipamento.

Os dois riscos são diferentes e exigem ações diferentes:

- **Risco financeiro** — o dinheiro não entra. Ação: cobrança, limite de crédito.
- **Risco de comodato** — a chopeira não volta. Ação: recolhimento, bloqueio de
  novo empréstimo.
- **Risco duplo** — as duas coisas ao mesmo tempo. É o grupo mais crítico: além
  de dever, o cliente está com um bem da empresa.

   
> **⚠️ INCONSISTÊNCIA DE FLAGS — CORRIGIDA APENAS PARA ESTA ANÁLISE**
>
> O consolidado traz **89 clientes** com parcelas a receber (`TEM_FINANCEIRO = 1`)
> ou equipamento cedido (`TEM_COMODATO = 1`), mas sem venda registrada
> (`TEM_VENDAS = 0`).
>
> **O que são esses 89.** Nenhum deles tem `FREQUENCIA_COMPRAS > 0`, e nenhum é
> `CORE_BUSINESS = 1`. São clientes com contrato financeiro ou comodato herdado,
> sem venda de chopp no recorte do notebook 01. A flag não está "errada" no
> sentido de contradizer a origem: `TEM_VENDAS` reflete a tabela de vendas, e
> essas linhas não estão nela.
>
> **O que a célula abaixo faz.** Cria `TEM_VENDAS_CORRIGIDO` = venda OU financeiro
> OU comodato, para que o gráfico de alcance conte *"clientes com alguma relação
> comercial"* (1.370) e não *"clientes que compraram"* (1.281). É uma definição
> mais larga para fins de retrato de carteira.
>
> **O que ela NÃO faz.** Não altera `FREQUENCIA_COMPRAS`, não altera
> `CORE_BUSINESS` e não muda nenhuma coluna que o notebook 04 usa. Como os 89 têm
> `CORE_BUSINESS = 0`, eles já ficam de fora do universo de modelagem pelo recorte
> de negócio — **esta correção não desloca em nada a população dos notebooks
> 04-07**. O funil da seção 10 usa as colunas originais, de propósito.


In [0]:
perfil = dados["PERFIL_RISCO"].value_counts().reindex(ORDEM_PERFIL_RISCO).fillna(0).astype(int)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2),
                               gridspec_kw={"width_ratios": [1.25, 1]})

barras = ax1.barh(list(perfil.index)[::-1], list(perfil.values)[::-1],
                  color=[CORES_PERFIL_RISCO[p] for p in list(perfil.index)[::-1]],
                  height=0.6)
ax1.xaxis.set_major_formatter(FuncFormatter(milhar))
ax1.set_xlim(0, perfil.max() * 1.2)
ax1.set_title("Clientes por perfil de risco")
ax1.grid(axis="y", visible=False)
rotular_barras(ax1, barras, list(perfil.values)[::-1])

# O dinheiro exposto importa mais que a contagem: 10 clientes grandes em risco
# pesam mais que 100 pequenos.
exposicao = dados.groupby("PERFIL_RISCO")["TOTAL_GASTO"].sum().reindex(
    ORDEM_PERFIL_RISCO).fillna(0)
barras2 = ax2.barh(list(exposicao.index)[::-1], list(exposicao.values)[::-1],
                   color=[CORES_PERFIL_RISCO[p] for p in list(exposicao.index)[::-1]],
                   height=0.6)
ax2.xaxis.set_major_formatter(FuncFormatter(moeda))
ax2.xaxis.set_major_locator(plt.MaxNLocator(4))   # 4 marcas: os rótulos R$ são largos
ax2.set_xlim(0, exposicao.max() * 1.30)
ax2.set_title("Faturamento associado a cada perfil")
ax2.set_yticks([])
ax2.grid(axis="y", visible=False)
rotular_barras(ax2, barras2, list(exposicao.values)[::-1], formato=moeda, folga=0.02)

plt.tight_layout()
plt.show()

em_risco = int(perfil.drop("SEM RISCO").sum())
duplo = int(perfil["RISCO DUPLO"])
valor_risco = float(exposicao.drop("SEM RISCO").sum())
leitura("O que isso significa", [
    f"{em_risco:,} clientes ({em_risco / total * 100:.0f}% da carteira) atrasam mais de 20%",
    "dos seus compromissos.",
    f"Esses clientes respondem por {moeda(valor_risco)} de faturamento "
    f"({valor_risco / dados['TOTAL_GASTO'].sum() * 100:.0f}% do total).",
    "",
    f"{duplo:,} estão em RISCO DUPLO: devem dinheiro E estão com equipamento da",
    "empresa. É por onde a cobrança deve começar — é o grupo em que a empresa",
    "tem, ao mesmo tempo, o maior prejuízo e o maior poder de negociação.",
])

## 4A. Evolução Temporal do Negócio

O comportamento da carteira muda ao longo do tempo. Algumas questões só ficam
visíveis quando se olha mês a mês:

- O volume de vendas e de comodatos cresce? Decresce? Há sazonalidade?
- O valor em atraso financeiro está aumentando ou diminuindo?
- Quantos equipamentos estão retidos (não devolvidos) a cada mês?

Estas séries históricas revelam tendências que um snapshot não mostra.

In [0]:
# ==============================================================================
# CARREGAMENTO DAS TABELAS TRANSACIONAIS COM CORREÇÕES
# ==============================================================================
# CORREÇÃO: Abandonar aproximações pela tabela consolidada `dados` e usar
# estritamente as tabelas transacionais do ERP para análise temporal precisa.

print("⏳ Carregando tabelas transacionais do Unity Catalog...")

# Carregar tabelas transacionais publicadas no UC
# (assumindo que foram ingeridas pelo notebook 02 ou pipeline ETL)
try:
    # Tentar carregar tabelas transacionais do Unity Catalog
    df_vendas_raw = spark.table("projetointegrador.projetointegrador.tb_pedido_item").toPandas()
    df_financeiro_raw = spark.table("projetointegrador.projetointegrador.tb_contas_a_receber_parcela").toPandas()
    df_comodato_raw = spark.table("projetointegrador.projetointegrador.tb_comodato_bem").toPandas()
    print("✅ Tabelas transacionais carregadas do Unity Catalog")
    DADOS_REAIS = True
except Exception as e:
    print(f"⚠️  Tabelas transacionais não encontradas no UC: {e}")
    print("⚠️  FALLBACK: Recriando a partir da tabela consolidada (limitações aplicam)")
    print("⚠️  Para análise precisa, publique as tabelas transacionais no UC.")
    
    # FALLBACK: Se as tabelas transacionais não existem, usar a tabela consolidada
    # com as mesmas limitações anteriores (documentadas)
    df_vendas_raw = None
    df_financeiro_raw = None
    df_comodato_raw = None
    DADOS_REAIS = False

# ==============================================================================
# DEFINIR FUNÇÕES DE CÁLCULO DE ATRASO
# ==============================================================================
def calc_atraso_financeiro(df_financeiro, data_corte):
    """
    Calcula atraso de pagamento para cada parcela baseado na data de corte.
    
    Args:
        df_financeiro: DataFrame com colunas DT_VENCIMENTO, DT_PAGAMENTO, VL_PARCELA
        data_corte: Data de referência para cálculo do atraso
    
    Returns:
        DataFrame com coluna ATRASOU_PAGAMENTO (1 = atrasou, 0 = não atrasou)
    """
    df = df_financeiro.copy()
    df['DT_VENCIMENTO'] = pd.to_datetime(df['DT_VENCIMENTO'], errors='coerce')
    df['DT_PAGAMENTO'] = pd.to_datetime(df['DT_PAGAMENTO'], errors='coerce')
    
    # Atraso = pagou após vencimento OU ainda não pagou e já venceu
    df['ATRASOU_PAGAMENTO'] = (
        ((df['DT_PAGAMENTO'].notna()) & (df['DT_PAGAMENTO'] > df['DT_VENCIMENTO'])) |
        ((df['DT_PAGAMENTO'].isna()) & (df['DT_VENCIMENTO'] < data_corte))
    ).astype(int)
    
    return df


def calc_atraso_comodato(df_comodato, data_corte):
    """
    Calcula atraso de devolução para cada equipamento baseado na data de corte.
    
    Args:
        df_comodato: DataFrame com colunas DT_VENCIMENTO, DT_RETORNO, QTD_PRODUTO
        data_corte: Data de referência para cálculo do atraso
    
    Returns:
        DataFrame com coluna ATRASOU_COMODATO (1 = atrasou, 0 = não atrasou)
    """
    df = df_comodato.copy()
    df['DT_VENCIMENTO'] = pd.to_datetime(df['DT_VENCIMENTO'], errors='coerce')
    df['DT_RETORNO'] = pd.to_datetime(df['DT_RETORNO'], errors='coerce')
    
    # Atraso = devolveu após vencimento OU ainda não devolveu e já venceu
    df['ATRASOU_COMODATO'] = (
        ((df['DT_RETORNO'].notna()) & (df['DT_RETORNO'] > df['DT_VENCIMENTO'])) |
        ((df['DT_RETORNO'].isna()) & (df['DT_VENCIMENTO'] < data_corte))
    ).astype(int)
    
    return df


# Data de corte (última transação conhecida)
data_corte = pd.to_datetime(dados['ULTIMA_COMPRA'].max())
print(f"Data de corte: {data_corte.strftime('%d/%m/%Y')}")

# ==============================================================================
# PROCESSAMENTO: VERIFICAR SE TEMOS DADOS TRANSACIONAIS REAIS
# ==============================================================================
if df_vendas_raw is not None and df_financeiro_raw is not None and df_comodato_raw is not None:
    print("\n📊 USANDO DADOS TRANSACIONAIS REAIS (tabelas detalhadas do ERP)")
    
    # --- GRÁFICO 1: Evolução Operacional Bruta ---
    # Vendas: agrupar por mês de emissão
    df_vendas_raw['DT_PEDIDO'] = pd.to_datetime(df_vendas_raw['DT_PEDIDO'], errors='coerce')
    df_vendas_raw['MES_ANO'] = df_vendas_raw['DT_PEDIDO'].dt.to_period('M')
    df_vendas_raw['VL_VENDA'] = pd.to_numeric(df_vendas_raw['VL_FINANCEIRO'], errors='coerce')
    
    vendas_mensais_valor = (
        df_vendas_raw.groupby('MES_ANO')['VL_VENDA']
        .sum()
        .sort_index()
    )
    
    # Comodatos: agrupar por mês de cessão
    df_comodato_raw['DT_EMISSAO'] = pd.to_datetime(df_comodato_raw['DT_EMISSAO'], errors='coerce')
    df_comodato_raw['MES_ANO'] = df_comodato_raw['DT_EMISSAO'].dt.to_period('M')
    df_comodato_raw['QTD_PRODUTO'] = pd.to_numeric(df_comodato_raw['QTD_PRODUTO'], errors='coerce')
    
    comodatos_mensais_qtd = (
        df_comodato_raw.groupby('MES_ANO')['QTD_PRODUTO']
        .sum()
        .sort_index()
    )
    
    # --- GRÁFICO 2: Atrasos Reais por Mês de Vencimento ---
    # CORREÇÃO CRÍTICA: Filtrar vencimentos até data_corte ANTES de calcular atrasos
    df_f_hist = calc_atraso_financeiro(df_financeiro_raw, data_corte)
    # Filtro essencial: apenas vencimentos até a data de corte
    df_f_hist = df_f_hist[pd.to_datetime(df_f_hist['DT_VENCIMENTO']) <= data_corte].copy()
    df_f_hist = df_f_hist[df_f_hist['ATRASOU_PAGAMENTO'] == 1].copy()
    df_f_hist['MES_ANO'] = pd.to_datetime(df_f_hist['DT_VENCIMENTO']).dt.to_period('M')
    hist_fin = df_f_hist.groupby('MES_ANO')['VL_PARCELA'].sum().reset_index()
    print(f"✓ Atrasos financeiros processados: {len(df_f_hist)} parcelas atrasadas")
    
    df_c_hist = calc_atraso_comodato(df_comodato_raw, data_corte)
    # Filtro essencial: apenas vencimentos até a data de corte
    df_c_hist = df_c_hist[pd.to_datetime(df_c_hist['DT_VENCIMENTO']) <= data_corte].copy()
    df_c_hist = df_c_hist[df_c_hist['ATRASOU_COMODATO'] == 1].copy()
    df_c_hist['MES_ANO'] = pd.to_datetime(df_c_hist['DT_VENCIMENTO']).dt.to_period('M')
    hist_com = df_c_hist.groupby('MES_ANO')['QTD_PRODUTO'].sum().reset_index()
    print(f"✓ Atrasos de comodato processados: {len(df_c_hist)} equipamentos retidos")
    
    df_hist = pd.merge(hist_fin, hist_com, on='MES_ANO', how='outer').fillna(0).sort_values('MES_ANO')
    
    valor_atraso_mensal = df_hist.set_index('MES_ANO')['VL_PARCELA']
    unidades_retidas_mensal = df_hist.set_index('MES_ANO')['QTD_PRODUTO']
    
    NOTA_LIMITACAO = ""  # Sem limitações - dados reais
    
else:
    print("\n⚠️  USANDO APROXIMAÇÕES (tabela consolidada por cliente)")
    print("⚠️  Limitação: Valores agregados por última compra, não por transação real.")
    
    # FALLBACK: Usar lógica anterior baseada na tabela consolidada
    clientes_ativos = dados[dados['TEM_VENDAS_CORRIGIDO'] == 1].copy()
    
    clientes_ativos['MES_ANO_PRIMEIRA'] = pd.to_datetime(
        clientes_ativos['PRIMEIRA_COMPRA'], errors='coerce'
    ).dt.to_period('M')
    
    # Gráfico 1: Contagem de novos clientes
    vendas_mensais_valor = (
        clientes_ativos.groupby('MES_ANO_PRIMEIRA')['TOTAL_GASTO']
        .sum()
        .sort_index()
    )
    
    comodatos_mensais_qtd = (
        clientes_ativos[clientes_ativos['TEM_COMODATO'] == 1]
        .groupby('MES_ANO_PRIMEIRA')['QTD_EQUIPAMENTOS']
        .sum()
        .sort_index()
    )
    
    # Gráfico 2: CORREÇÃO - Distribuir atrasos ao longo do histórico do cliente
    # Em vez de concentrar tudo na ULTIMA_COMPRA, distribuir proporcionalmente
    # entre PRIMEIRA_COMPRA e ULTIMA_COMPRA
    
    clientes_risco = dados[dados['RISCO_FINANCEIRO'] == 1].copy()
    
    # Criar distribuição temporal dos atrasos financeiros
    rows_financeiro = []
    for _, cliente in clientes_risco.iterrows():
        primeira = pd.to_datetime(cliente['PRIMEIRA_COMPRA'])
        ultima = pd.to_datetime(cliente['ULTIMA_COMPRA'])
        
        if pd.notna(primeira) and pd.notna(ultima) and primeira <= ultima:
            # Distribuir o valor proporcionalmente ao longo dos meses ativos
            meses_ativos = pd.period_range(start=primeira.to_period('M'), 
                                          end=ultima.to_period('M'), freq='M')
            n_meses = len(meses_ativos)
            valor_por_mes = cliente['VALOR_TOTAL_PARCELAS'] / n_meses if n_meses > 0 else 0
            
            for mes in meses_ativos:
                rows_financeiro.append({'MES_ANO': mes, 'VALOR': valor_por_mes})
    
    if rows_financeiro:
        df_fin_dist = pd.DataFrame(rows_financeiro)
        valor_atraso_mensal = df_fin_dist.groupby('MES_ANO')['VALOR'].sum().sort_index()
    else:
        valor_atraso_mensal = pd.Series(dtype=float)
    
    # Mesma lógica para comodatos
    clientes_comodato_risco = dados[dados['RISCO_COMODATO'] == 1].copy()
    
    rows_comodato = []
    for _, cliente in clientes_comodato_risco.iterrows():
        primeira = pd.to_datetime(cliente['PRIMEIRA_COMPRA'])
        ultima = pd.to_datetime(cliente['ULTIMA_COMPRA'])
        
        if pd.notna(primeira) and pd.notna(ultima) and primeira <= ultima:
            meses_ativos = pd.period_range(start=primeira.to_period('M'), 
                                          end=ultima.to_period('M'), freq='M')
            n_meses = len(meses_ativos)
            qtd_por_mes = cliente['QTD_EQUIPAMENTOS'] / n_meses if n_meses > 0 else 0
            
            for mes in meses_ativos:
                rows_comodato.append({'MES_ANO': mes, 'QTD': qtd_por_mes})
    
    if rows_comodato:
        df_com_dist = pd.DataFrame(rows_comodato)
        unidades_retidas_mensal = df_com_dist.groupby('MES_ANO')['QTD'].sum().sort_index()
    else:
        unidades_retidas_mensal = pd.Series(dtype=float)
    
    NOTA_LIMITACAO = (
        "\n⚠️  NOTA: Gráficos baseados em APROXIMAÇÕES (tabela consolidada).\n"
        "    Para análise precisa, publique as tabelas transacionais no Unity Catalog:\n"
        "    - projetointegrador.projetointegrador.tb_pedido_item\n"
        "    - projetointegrador.projetointegrador.tb_contas_a_receber_parcela\n"
        "    - projetointegrador.projetointegrador.tb_comodato_bem"
    )

print("\n✓ Séries temporais calculadas")

# ==============================================================================
# VISUALIZAÇÃO
# ==============================================================================
# Alinhar índices para garantir mesmo período
all_months_g1 = vendas_mensais_valor.index.union(comodatos_mensais_qtd.index)
vendas_mensais_valor = vendas_mensais_valor.reindex(all_months_g1, fill_value=0)
comodatos_mensais_qtd = comodatos_mensais_qtd.reindex(all_months_g1, fill_value=0)

all_months_g2 = valor_atraso_mensal.index.union(unidades_retidas_mensal.index)
valor_atraso_mensal = valor_atraso_mensal.reindex(all_months_g2, fill_value=0)
unidades_retidas_mensal = unidades_retidas_mensal.reindex(all_months_g2, fill_value=0)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# ------------------------------------------------------------------------------
# GRÁFICO 1: EVOLUÇÃO OPERACIONAL BRUTA (VENDAS R$ vs COMODATO QTD)
# ------------------------------------------------------------------------------
mes_labels_g1 = [str(m) for m in all_months_g1]
x_pos_g1 = range(len(mes_labels_g1))

# Criar eixo Y secundário para quantidade de comodatos
ax1_twin = ax1.twinx()

# Vendas em R$ (barras no eixo Y esquerdo) - COR SALMÃO/VERMELHO
barras_vendas = ax1.bar(x_pos_g1, vendas_mensais_valor.values,
                        width=0.7, label='Faturamento Vendas (R$)', 
                        color='#e57373', alpha=0.8)

# Comodatos em quantidade (linha no eixo Y direito) - COR LARANJA
linha_comodatos = ax1_twin.plot(x_pos_g1, comodatos_mensais_qtd.values,
                                label='Quantidade Equipamentos', 
                                color='#ff9800', linewidth=2.5, marker='o', markersize=6)

ax1.set_xticks(x_pos_g1[::max(1, len(x_pos_g1)//12)])  # Mostrar até 12 labels
ax1.set_xticklabels([mes_labels_g1[i] for i in range(0, len(mes_labels_g1), max(1, len(mes_labels_g1)//12))],
                    rotation=45, ha='right', fontsize=8.5)
ax1.set_title('Gráfico 1: Evolução Operacional Bruta (Faturamento vs. Equipamentos Cedidos)', 
             fontweight='bold', fontsize=11.5, pad=15)
ax1.set_xlabel('Mês', fontsize=10)
ax1.set_ylabel('Faturamento Vendas (R$)', fontsize=10, color='#e57373')
ax1_twin.set_ylabel('Quantidade de Equipamentos', fontsize=10, color='#ff9800')
ax1.tick_params(axis='y', labelcolor='#e57373')
ax1_twin.tick_params(axis='y', labelcolor='#ff9800')
ax1.yaxis.set_major_formatter(FuncFormatter(moeda))
ax1_twin.yaxis.set_major_formatter(FuncFormatter(milhar))
ax1.grid(axis='x', visible=False)

# Legenda combinada
linhas_labels = [barras_vendas, linha_comodatos[0]]
labels = ['Faturamento Vendas (R$)', 'Quantidade Equipamentos']
ax1.legend(linhas_labels, labels, loc='upper left', fontsize=9.5)

# Rótulos em TODAS as barras (sem filtro)
for i, (barra, valor) in enumerate(zip(barras_vendas, vendas_mensais_valor.values)):
    if valor > 0:  # Mostrar apenas se houver valor
        ax1.text(barra.get_x() + barra.get_width()/2, valor + ax1.get_ylim()[1] * 0.01,
                moeda(valor), ha='center', va='bottom', fontsize=7, color='#e57373', fontweight='bold')

# Rótulos em TODOS os pontos da linha de comodato
for i, (x, valor) in enumerate(zip(x_pos_g1, comodatos_mensais_qtd.values)):
    if valor > 0:
        ax1_twin.text(x, valor + ax1_twin.get_ylim()[1] * 0.02,
                     f'{int(valor)}', ha='center', va='bottom', fontsize=7, 
                     color='#ff9800', fontweight='bold')

# ------------------------------------------------------------------------------
# GRÁFICO 2: DUAL AXIS (VALOR EM ATRASO VS UNIDADES RETIDAS)
# ------------------------------------------------------------------------------
mes_labels_2 = [str(m) for m in valor_atraso_mensal.index]
x_pos_2 = range(len(mes_labels_2))

# Eixo Y esquerdo: Valor em Atraso (Barras salmão/vermelho)
color_valor = '#e57373'  # Salmão/vermelho
barras_valor = ax2.bar(x_pos_2, valor_atraso_mensal.values, 
                       color=color_valor, alpha=0.75, width=0.6,
                       label='R$ em Atraso Financeiro')
ax2.set_xlabel('Mês', fontsize=10)
ax2.set_ylabel('Valor em Atraso Financeiro (R$)', fontsize=10, color=color_valor)
ax2.tick_params(axis='y', labelcolor=color_valor)
ax2.yaxis.set_major_formatter(FuncFormatter(moeda))

# Rótulos nas barras
for i, (x, valor) in enumerate(zip(x_pos_2, valor_atraso_mensal.values)):
    if valor > 0:
        label = moeda(valor).replace('R$ ', 'R$ ')
        ax2.text(x, valor + ax2.get_ylim()[1] * 0.02,
                label, ha='center', va='bottom', fontsize=7.5, 
                color=color_valor, fontweight='bold')

# Eixo Y direito: Unidades Retidas (Linha laranja com marcadores)
ax2_right = ax2.twinx()
color_unidades = '#ff9800'  # Laranja
linha_unidades = ax2_right.plot(x_pos_2, unidades_retidas_mensal.values,
                                color=color_unidades, marker='o', linewidth=2.5,
                                markersize=7, label='Unidades Retidas (Comodato)',
                                alpha=0.9)
ax2_right.set_ylabel('Unidades Retidas (Comodato)', fontsize=10, color=color_unidades)
ax2_right.tick_params(axis='y', labelcolor=color_unidades)
ax2_right.yaxis.set_major_formatter(FuncFormatter(milhar))

# Rótulos nos pontos da linha
for i, (x, valor) in enumerate(zip(x_pos_2, unidades_retidas_mensal.values)):
    if valor > 0:
        ax2_right.text(x, valor + ax2_right.get_ylim()[1] * 0.03,
                      f'{int(valor)}', ha='center', va='bottom', fontsize=7.5,
                      color=color_unidades, fontweight='bold')

ax2.set_xticks(x_pos_2)
ax2.set_xticklabels(mes_labels_2, rotation=45, ha='right', fontsize=8.5)
ax2.set_title('Série Histórica: Impacto Financeiro vs. Retenção de Ativos por Mês',
             fontweight='bold', fontsize=11.5)
ax2.grid(axis='x', visible=False)
ax2.grid(axis='y', linestyle='--', alpha=0.4)

# Legenda integrada (combinar ambas as séries)
linhas_legendas = [barras_valor, linha_unidades[0]]
labels_legendas = ['R$ em Atraso Financeiro', 'Unidades Retidas (Comodato)']
ax2.legend(linhas_legendas, labels_legendas, loc='upper left', fontsize=9)

plt.tight_layout()
plt.show()

if NOTA_LIMITACAO:
    print(NOTA_LIMITACAO)

# ==============================================================================
# INTERPRETAÇÃO DOS GRÁFICOS
# ==============================================================================
mes_labels_g1 = [str(m) for m in all_months_g1]
mes_labels_g2 = [str(m) for m in all_months_g2]

if len(vendas_mensais_valor) > 0 and vendas_mensais_valor.max() > 0:
    idx_max_vendas = vendas_mensais_valor.values.argmax()
    mes_max_vendas = mes_labels_g1[idx_max_vendas]
    val_max_vendas = vendas_mensais_valor.values[idx_max_vendas]
else:
    mes_max_vendas = "N/A"
    val_max_vendas = 0

if len(comodatos_mensais_qtd) > 0 and comodatos_mensais_qtd.max() > 0:
    idx_max_comodatos = comodatos_mensais_qtd.values.argmax()
    mes_max_comodatos = mes_labels_g1[idx_max_comodatos]
    val_max_comodatos = comodatos_mensais_qtd.values[idx_max_comodatos]
else:
    mes_max_comodatos = "N/A"
    val_max_comodatos = 0

if len(valor_atraso_mensal) > 0 and valor_atraso_mensal.max() > 0:
    idx_max_atraso = valor_atraso_mensal.values.argmax()
    mes_max_atraso = mes_labels_g2[idx_max_atraso]
    val_max_atraso = valor_atraso_mensal.values[idx_max_atraso]
else:
    mes_max_atraso = "N/A"
    val_max_atraso = 0

if len(unidades_retidas_mensal) > 0 and unidades_retidas_mensal.max() > 0:
    idx_max_retidos = unidades_retidas_mensal.values.argmax()
    mes_max_retidos = mes_labels_g2[idx_max_retidos]
    val_max_retidos = int(unidades_retidas_mensal.values[idx_max_retidos])
else:
    mes_max_retidos = "N/A"
    val_max_retidos = 0

leitura("O que isso significa", [
    f"GRÁFICO 1 (Evolução Operacional Bruta): Mostra {'fluxo real de vendas (R$) e equipamentos' if DADOS_REAIS else 'aproximação por novos clientes'}.",
    f"  • Pico de faturamento: {mes_max_vendas} com {moeda(val_max_vendas)}.",
    f"  • Pico de comodatos: {mes_max_comodatos} com {int(val_max_comodatos)} equipamentos cedidos.",
    "",
    f"GRÁFICO 2 (Atrasos Mensais): Compara impacto financeiro com retenção de ativos.",
    f"  • Mês com maior valor em atraso: {mes_max_atraso} ({moeda(val_max_atraso)}).",
    f"  • Mês com mais equipamentos retidos: {mes_max_retidos} ({val_max_retidos} unidades).",
    "",
    "METODOLOGIA: " + ("Dados TRANSACIONAIS REAIS (por transação/vencimento)." if DADOS_REAIS else
                      "APROXIMAÇÕES (consolidado por cliente, não transacional)."),
    "" if DADOS_REAIS else "Para análise precisa, publique as tabelas transacionais no Unity Catalog.",
])

## 5. Há quanto tempo os atrasos se arrastam

Atraso de 2 dias e atraso de 6 meses são problemas diferentes. O primeiro é
esquecimento; o segundo, na prática, é perda.

O gráfico abaixo classifica cada cliente pela **pior** ocorrência do seu
histórico — não pela média. Um cliente que atrasou 45 dias uma vez é um caso de
"+30 Dias", ainda que a média o colocasse numa faixa branda: a média esconde
exatamente o evento que interessa.

In [0]:
fig, eixos = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)

for ax, coluna, titulo, universo in [
    (eixos[0], "AGING_PAGAMENTO", "Atraso de pagamento", dados["TEM_FINANCEIRO"] == 1),
    (eixos[1], "AGING_COMODATO", "Atraso na devolução", dados["TEM_COMODATO"] == 1),
]:
    # Só quem tem a trilha entra na conta: cliente sem comodato não é "sem
    # atraso de comodato", ele simplesmente não tem comodato.
    faixas = (dados[universo][coluna].value_counts()
              .reindex(ORDEM_AGING).fillna(0).astype(int))
    cores = [cor_severidade(i, len(ORDEM_AGING)) for i in range(len(ORDEM_AGING))]
    barras = ax.bar(range(len(faixas)), faixas.values, color=cores, width=0.68)
    ax.set_xticks(range(len(faixas)))
    ax.set_xticklabels(faixas.index, rotation=45, ha="right", fontsize=9)
    ax.set_title(f"{titulo}  ({int(universo.sum()):,} clientes)")
    ax.yaxis.set_major_formatter(FuncFormatter(milhar))
    ax.grid(axis="x", visible=False)
    for barra, valor in zip(barras, faixas.values):
        if valor:
            ax.text(barra.get_x() + barra.get_width() / 2, valor + faixas.max() * 0.02,
                    milhar(valor), ha="center", fontsize=8.5, color=COR_TEXTO)
    ax.set_ylim(0, faixas.max() * 1.15)

eixos[0].set_ylabel("clientes")
plt.tight_layout()
plt.show()

com_fin = dados[dados["TEM_FINANCEIRO"] == 1]
com_com = dados[dados["TEM_COMODATO"] == 1]
graves_fin = int((com_fin["AGING_PAGAMENTO"] == "+30 Dias").sum())
graves_com = int((com_com["AGING_COMODATO"] == "+30 Dias").sum())
leitura("O que isso significa", [
    f"Pagamento: {graves_fin:,} clientes já atrasaram mais de 30 dias "
    f"({graves_fin / max(len(com_fin), 1) * 100:.0f}% dos que têm parcelas).",
    f"Comodato: {graves_com:,} clientes ficaram mais de 30 dias com o equipamento",
    f"além do prazo ({graves_com / max(len(com_com), 1) * 100:.0f}% dos que têm comodato).",
    "",
    "As barras à direita são as que exigem ação. Quanto mais à direita, menor a",
    "chance de recuperação espontânea — e maior o custo de continuar esperando.",
])

## 6. O risco tem endereço, forma de pagamento e tipo de negócio?

Se clientes de uma certa cidade, de um certo tipo de estabelecimento ou de uma
certa forma de pagamento atrasam sistematicamente mais, isso é acionável: dá para
ajustar política de crédito por grupo, sem esperar o cliente atrasar.

A linha tracejada é a média da carteira. Barra acima da linha = grupo pior que a
média. Só entram grupos com pelo menos 30 clientes — abaixo disso, a diferença
pode ser sorte.

In [0]:
MINIMO_GRUPO = 30
media_geral = dados["RISCO_FINANCEIRO"].mean() * 100

fig, eixos = plt.subplots(1, 3, figsize=(13.5, 4.2))

for ax, dimensao, titulo in [
    (eixos[0], "CIDADE", "Por cidade"),
    (eixos[1], "PAGAMENTO", "Por forma de pagamento"),
    (eixos[2], "PERFIL", "Por perfil da pessoa"),
]:
    grupo = dados.groupby(dimensao).agg(
        clientes=("ID_PESSOA", "count"),
        risco=("RISCO_FINANCEIRO", "mean"),
    )
    grupo = grupo[grupo["clientes"] >= MINIMO_GRUPO].copy()
    grupo["risco"] *= 100
    grupo = grupo.sort_values("risco").tail(8)

    if grupo.empty:
        ax.text(0.5, 0.5, f"nenhum grupo com {MINIMO_GRUPO}+ clientes",
                ha="center", va="center", color=COR_APOIO, transform=ax.transAxes)
        ax.set_title(titulo)
        ax.set_axis_off()
        continue

    # Emphasis: o grupo pior que a média ganha cor; o resto é contexto.
    cores = [RAMPA_SEVERIDADE[-2] if v > media_geral else COR_NEUTRA
             for v in grupo["risco"]]
    rotulos = [f"{i[:22]}  (n={int(n)})" for i, n in zip(grupo.index, grupo["clientes"])]
    barras = ax.barh(rotulos, grupo["risco"].values, color=cores, height=0.62)
    ax.axvline(media_geral, color=COR_TEXTO, linestyle="--", linewidth=1.2)
    ax.set_xlim(0, max(grupo["risco"].max() * 1.25, media_geral * 1.3))
    ax.set_title(titulo)
    ax.set_xlabel("% de clientes em risco financeiro")
    ax.grid(axis="y", visible=False)
    ax.tick_params(axis="y", labelsize=8.5)
    for barra, valor in zip(barras, grupo["risco"].values):
        ax.text(valor + ax.get_xlim()[1] * 0.035, barra.get_y() + barra.get_height() / 2,
                f"{valor:.0f}%", va="center", fontsize=8.5, color=COR_TEXTO)

fig.suptitle(f"Risco financeiro por grupo — média da carteira: {media_geral:.0f}% "
             "(linha tracejada)", fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

leitura("O que isso significa", [
    f"A média da carteira é {media_geral:.0f}% de clientes em risco financeiro.",
    "Os grupos coloridos ficam acima dessa média — são candidatos a uma política",
    "de crédito mais rígida (entrada maior, limite menor, prazo mais curto).",
    "",
    "Cuidado ao ler: grupo pequeno oscila muito. Por isso só aparecem aqui os",
    f"grupos com {MINIMO_GRUPO} clientes ou mais, e o n de cada um está no rótulo.",
    "Nenhuma dessas diferenças foi testada estatisticamente — são indícios para",
    "investigar, não conclusões fechadas.",
])

## 7. Cliente que sumiu é cliente que não pagou?

Uma suspeita comum na operação: o cliente que parou de comprar parou porque está
devendo. Se for verdade, o tempo desde a última compra vira um sinal de alerta
barato — não exige cálculo nenhum, está no extrato.

O gráfico testa a suspeita. Ele cruza há quanto tempo o cliente não compra com o
percentual dele que está em risco — e o resultado pode contrariar a intuição.

In [0]:
com_compra = dados[dados["TEM_VENDAS"] == 1].copy()

FAIXAS = [(0, 30, "até 1 mês"), (30, 90, "1 a 3 meses"), (90, 180, "3 a 6 meses"),
          (180, 365, "6 a 12 meses"), (365, np.inf, "mais de 1 ano")]
com_compra["RECENCIA"] = pd.cut(
    com_compra["DIAS_DESDE_ULTIMA_COMPRA"],
    bins=[f[0] for f in FAIXAS] + [np.inf],
    labels=[f[2] for f in FAIXAS], right=False,
)

recencia = com_compra.groupby("RECENCIA", observed=False).agg(
    clientes=("ID_PESSOA", "count"),
    risco_financeiro=("RISCO_FINANCEIRO", "mean"),
    risco_comodato=("RISCO_COMODATO", "mean"),
)
recencia[["risco_financeiro", "risco_comodato"]] *= 100

fig, ax = plt.subplots(figsize=(10, 4.2))
x = np.arange(len(recencia))
largura = 0.38
ax.bar(x - largura / 2, recencia["risco_financeiro"], largura,
       label="Risco financeiro (não paga)", color=COR_FINANCEIRO)
ax.bar(x + largura / 2, recencia["risco_comodato"], largura,
       label="Risco de comodato (não devolve)", color=COR_COMODATO)

for i, (fin, com, n) in enumerate(zip(recencia["risco_financeiro"],
                                      recencia["risco_comodato"],
                                      recencia["clientes"])):
    ax.text(i - largura / 2, fin + 1.5, f"{fin:.0f}%", ha="center", fontsize=8.5,
            color=COR_TEXTO)
    ax.text(i + largura / 2, com + 1.5, f"{com:.0f}%", ha="center", fontsize=8.5,
            color=COR_TEXTO)

ax.set_xticks(x)
ax.set_xticklabels([f"{r}\n(n={int(n)})"
                    for r, n in zip(recencia.index, recencia["clientes"])])
ax.set_xlabel("tempo desde a última compra")
ax.set_ylabel("% de clientes em risco")
ax.set_title("O cliente que sumiu está devendo?")
ax.set_ylim(0, max(recencia[["risco_financeiro", "risco_comodato"]].max()) * 1.25)
ax.legend(frameon=False, loc="upper left")
ax.grid(axis="x", visible=False)
plt.tight_layout()
plt.show()

recentes = recencia["risco_financeiro"].iloc[0]
antigos = recencia["risco_financeiro"].iloc[-1]
sobe = antigos > recentes
leitura("O que isso significa", [
    f"Quem comprou {recencia.index[0]}: {recentes:.0f}% em risco financeiro.",
    f"Quem não compra há {recencia.index[-1]}: {antigos:.0f}%.",
    "",
    ("O risco CRESCE conforme o cliente some — sumiço e inadimplência andam"
     if sobe else
     "O risco CAI conforme o cliente some — o oposto da suspeita comum:"),
    ("juntos, e o tempo sem comprar serve de alerta precoce." if sobe else
     "quem ainda compra é justamente quem ainda tem conta aberta para atrasar."),
    "",
    ("" if sobe else
     "Faz sentido operacional: cliente ativo compra a prazo toda semana e tem"),
    ("" if sobe else
     "muitas parcelas correndo; quem parou já liquidou ou foi cortado. Ou seja,"),
    ("" if sobe else
     "tempo sem comprar NÃO serve de alerta de inadimplência nesta carteira."),
    "",
    "Em nenhum dos casos o dado mostra causa: não se sabe se o cliente sumiu",
    "porque devia, ou se deve porque sumiu. Serve para priorizar visita — não",
    "para afirmar o motivo.",
])

## 8. Comprar a prazo prevê inadimplência?

O notebook 01 passou a trazer, cliente a cliente, **que fração das compras foi
contratada a prazo** — não o desfecho do pagamento, o termo combinado na venda.
É informação que existe antes de qualquer atraso acontecer, então serve para
prever, não só para descrever.

O gráfico separa a carteira em quatro grupos por esse percentual.

In [0]:
FAIXAS_PRAZO = [(-0.001, 0.001, "0%\n(só à vista)"), (0.001, 0.5, "1–50%"),
               (0.5, 0.999, "51–99%"), (0.999, 1.001, "100%\n(só a prazo)")]
dados["FAIXA_A_PRAZO"] = pd.cut(
    dados["PCT_COMPRAS_A_PRAZO"],
    bins=[f[0] for f in FAIXAS_PRAZO] + [FAIXAS_PRAZO[-1][1]],
    labels=[f[2] for f in FAIXAS_PRAZO], include_lowest=True,
)

prazo = dados.groupby("FAIXA_A_PRAZO", observed=False).agg(
    clientes=("ID_PESSOA", "count"), risco=("RISCO_FINANCEIRO", "mean"),
)
prazo["risco"] *= 100

fig, ax = plt.subplots(figsize=(9, 4.2))
cores = [cor_severidade(i, len(prazo)) for i in range(len(prazo))]
barras = ax.bar(range(len(prazo)), prazo["risco"], color=cores, width=0.62)
ax.set_xticks(range(len(prazo)))
ax.set_xticklabels(prazo.index, fontsize=9.5)
ax.set_ylabel("% de clientes em risco financeiro")
ax.set_xlabel("fração das compras contratada a prazo")
ax.set_title("Risco financeiro por mix de forma de pagamento")
ax.set_ylim(0, prazo["risco"].max() * 1.22)
ax.grid(axis="x", visible=False)
for barra, valor, n in zip(barras, prazo["risco"], prazo["clientes"]):
    ax.text(barra.get_x() + barra.get_width() / 2, valor + prazo["risco"].max() * 0.03,
            f"{valor:.0f}%\n(n={int(n)})", ha="center", fontsize=8.5, color=COR_TEXTO)
plt.tight_layout()
plt.show()

so_vista = prazo["risco"].iloc[0]
pico = prazo["risco"].max()
faixa_pico = prazo["risco"].idxmax()
leitura("O que isso significa", [
    f"Cliente que só compra à vista: {so_vista:.0f}% em risco financeiro — o grupo",
    "mais seguro da carteira, por construção (não tem parcela para atrasar).",
    "",
    f"O pico não é em quem compra 100% a prazo: é na faixa '{faixa_pico}',",
    f"com {pico:.0f}% em risco. Cliente que MISTURA formas de pagamento é o mais",
    "arriscado — mais que quem compra só a prazo de forma consistente.",
    "",
    "Leitura prática: a régua de crédito não deveria olhar só 'compra a prazo?'",
    "(sim/não). O padrão de USO da forma de pagamento é o que discrimina risco.",
])

## 9. Onde está o dinheiro em jogo

As análises anteriores contam clientes. Esta conta **reais**.

Os dois lados importam: um cliente pequeno que nunca paga é um aborrecimento; um
cliente grande que atrasa é um problema de caixa. O gráfico separa os clientes em
quatro grupos de tamanho igual (25% cada) pelo faturamento, e mostra o risco em
cada um.

In [0]:
com_faturamento = dados[dados["TOTAL_GASTO"] > 0].copy()
com_faturamento["PORTE"] = pd.qcut(
    com_faturamento["TOTAL_GASTO"], 4,
    labels=["25% menores", "25% médio-baixo", "25% médio-alto", "25% maiores"],
)

porte = com_faturamento.groupby("PORTE", observed=False).agg(
    clientes=("ID_PESSOA", "count"),
    faturamento=("TOTAL_GASTO", "sum"),
    risco=("RISCO_FINANCEIRO", "mean"),
)
porte["risco"] *= 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2))

cores = [cor_severidade(i, len(porte)) for i in range(len(porte))]
barras = ax1.bar(range(len(porte)), porte["faturamento"], color=cores, width=0.66)
ax1.set_xticks(range(len(porte)))
ax1.set_xticklabels(porte.index, rotation=20, ha="right", fontsize=9)
ax1.yaxis.set_major_formatter(FuncFormatter(moeda))
ax1.set_title("Faturamento concentrado por porte de cliente")
ax1.grid(axis="x", visible=False)
for barra, valor in zip(barras, porte["faturamento"]):
    ax1.text(barra.get_x() + barra.get_width() / 2, valor + porte["faturamento"].max() * 0.02,
             moeda(valor), ha="center", fontsize=8.5, color=COR_TEXTO)
ax1.set_ylim(0, porte["faturamento"].max() * 1.15)

barras2 = ax2.bar(range(len(porte)), porte["risco"], color=cores, width=0.66)
ax2.set_xticks(range(len(porte)))
ax2.set_xticklabels(porte.index, rotation=20, ha="right", fontsize=9)
ax2.set_title("% em risco financeiro dentro de cada grupo")
ax2.set_ylabel("% de clientes em risco")
ax2.grid(axis="x", visible=False)
for barra, valor in zip(barras2, porte["risco"]):
    ax2.text(barra.get_x() + barra.get_width() / 2, valor + porte["risco"].max() * 0.03,
             f"{valor:.0f}%", ha="center", fontsize=8.5, color=COR_TEXTO)
ax2.set_ylim(0, porte["risco"].max() * 1.2)

plt.tight_layout()
plt.show()

topo = porte.iloc[-1]
fatia_topo = topo["faturamento"] / porte["faturamento"].sum() * 100
leitura("O que isso significa", [
    f"Os 25% maiores clientes concentram {fatia_topo:.0f}% do faturamento.",
    f"Dentro desse grupo, {topo['risco']:.0f}% estão em risco financeiro.",
    "",
    "É onde a cobrança rende mais por telefonema: são poucos clientes e muito",
    "dinheiro. Perder um cliente grande custa mais do que perder dezenas de",
    "pequenos — e, no grupo de cima, a régua de crédito não deveria ser a mesma.",
])

### 9.1 O mesmo raciocínio vale para o equipamento cedido

`TOTAL_GASTO` é o dinheiro que já passou pelo caixa. O valor do equipamento em
comodato é dinheiro que **ainda está na rua** — e o notebook 01 agora traz o
valor médio acordado por contrato (`VALOR_MEDIO_COMODATO`), não só a contagem de
equipamentos.

In [0]:
com_comodato = dados[dados["TEM_COMODATO"] == 1].copy()
# duplicates="drop": muitos contratos têm o MESMO valor acordado (ex.: preço de
# tabela padrão), então os cortes de quartil colidem e sobram menos de 4 faixas.
# Rotular por posição em vez de uma lista fixa de 4 evita o ValueError do qcut.
_faixa_valor, _bordas = pd.qcut(
    com_comodato["VALOR_MEDIO_COMODATO"], 4, duplicates="drop", retbins=True
)
_rotulos_valor = [f"{i + 1}ª faixa de valor" for i in range(len(_bordas) - 1)]
_rotulos_valor[0], _rotulos_valor[-1] = "menor valor", "maior valor"
com_comodato["FAIXA_VALOR"] = _faixa_valor.cat.rename_categories(_rotulos_valor)

valor_com = com_comodato.groupby("FAIXA_VALOR", observed=False).agg(
    clientes=("ID_PESSOA", "count"),
    valor_exposto=("VALOR_MEDIO_COMODATO", "sum"),
    risco=("RISCO_COMODATO", "mean"),
)
valor_com["risco"] *= 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2))
cores = [cor_severidade(i, len(valor_com)) for i in range(len(valor_com))]

barras = ax1.bar(range(len(valor_com)), valor_com["valor_exposto"], color=cores, width=0.66)
ax1.set_xticks(range(len(valor_com)))
ax1.set_xticklabels(valor_com.index, rotation=20, ha="right", fontsize=9)
ax1.yaxis.set_major_formatter(FuncFormatter(moeda))
ax1.set_title("Valor de equipamento exposto por faixa")
ax1.grid(axis="x", visible=False)
for barra, valor in zip(barras, valor_com["valor_exposto"]):
    ax1.text(barra.get_x() + barra.get_width() / 2, valor + valor_com["valor_exposto"].max() * 0.02,
             moeda(valor), ha="center", fontsize=8.5, color=COR_TEXTO)
ax1.set_ylim(0, valor_com["valor_exposto"].max() * 1.15)

barras2 = ax2.bar(range(len(valor_com)), valor_com["risco"], color=cores, width=0.66)
ax2.set_xticks(range(len(valor_com)))
ax2.set_xticklabels(valor_com.index, rotation=20, ha="right", fontsize=9)
ax2.set_title("% em risco de comodato dentro de cada faixa")
ax2.set_ylabel("% de clientes em risco")
ax2.grid(axis="x", visible=False)
for barra, valor in zip(barras2, valor_com["risco"]):
    ax2.text(barra.get_x() + barra.get_width() / 2, valor + valor_com["risco"].max() * 0.03,
             f"{valor:.0f}%", ha="center", fontsize=8.5, color=COR_TEXTO)
ax2.set_ylim(0, valor_com["risco"].max() * 1.2)

plt.tight_layout()
plt.show()

topo_valor = valor_com.iloc[-1]
fatia_valor = topo_valor["valor_exposto"] / valor_com["valor_exposto"].sum() * 100
leitura("O que isso significa", [
    f"A faixa de maior valor concentra {fatia_valor:.0f}% de todo o equipamento",
    f"exposto — e é também a faixa com MAIS risco de não devolução "
    f"({topo_valor['risco']:.0f}%).",
    "",
    "É o pior cenário possível: onde há mais chopeira em jogo é também onde há",
    "mais chance de não voltar. Contratos de alto valor merecem garantia extra",
    "(fiador, caução) — não só um limite de crédito mais alto.",
])

## 10. O funil: de quem está no ERP a quem entra no modelo

> **A pergunta:** quantos clientes o pipeline descarta em cada etapa, e por quê?

O consolidado já chega filtrado — o notebook 01 aplica as regras do ERP
(`FL_ATIVO`, tipo de operação, pessoa que é cliente). O que sobra aqui são os
recortes **analíticos**: o de negócio (comprou chopp) e o de histórico mínimo.

Cada degrau é uma decisão que alguém tomou. O funil torna essas decisões visíveis
e reversíveis, em vez de deixá-las implícitas no meio do código.

In [0]:
etapas = [
    ("Consolidado (notebook 01)", len(dados),
     "já sem inativos, sem cancelados e sem não-clientes"),
    ("Com histórico de compra", int((dados["FREQUENCIA_COMPRAS"] > 0).sum()),
     "quem nunca comprou não tem RFM"),
    ("Core business", int(dados["CORE_BUSINESS"].sum()),
     "comprou chopp, chopeira ou barril"),
]
for minimo in [1, 2]:
    n = int(((dados["CORE_BUSINESS"] == 1) & (dados["FREQUENCIA_COMPRAS"] > minimo)).sum())
    etapas.append((f"Core + {minimo + 1}+ compras", n,
                   "histórico mínimo para a taxa significar algo"))

base = etapas[0][1]
print(f"{'etapa':<32}{'clientes':>10}{'retenção':>11}{'perda':>9}")
print("-" * 68)
anterior = base
for nome, n, _ in etapas:
    print(f"{nome:<32}{n:>10,}{n / base * 100:>10.1f}%{n - anterior:>9,}")
    anterior = n

print()
for nome, n, motivo in etapas[1:]:
    print(f"  {nome:<32} {motivo}")

fig, ax = plt.subplots(figsize=(9, 3.4))
nomes = [e[0] for e in etapas][::-1]
valores = [e[1] for e in etapas][::-1]
barras = ax.barh(nomes, valores, color=RAMPA_SEVERIDADE[:len(etapas)][::-1], height=0.62)
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.set_xlim(0, base * 1.22)
ax.set_title("Do consolidado ao universo de modelagem", fontsize=11, fontweight="bold")
for barra, valor in zip(barras, valores):
    ax.text(barra.get_width() + base * 0.02, barra.get_y() + barra.get_height() / 2,
            f"{valor:,} ({valor / base * 100:.0f}%)", va="center", fontsize=9)
plt.tight_layout()
plt.show()

corte_atual = int(((dados["CORE_BUSINESS"] == 1) & (dados["FREQUENCIA_COMPRAS"] > 0)).sum())
n_com_compra = int((dados["FREQUENCIA_COMPRAS"] > 0).sum())
n_core = int(dados["CORE_BUSINESS"].sum())
leitura("O que isso significa", [
    f"{len(dados) - n_com_compra:,} clientes não têm compra alguma e saem primeiro.",
    f"Dos {n_com_compra:,} que compraram, {n_com_compra - n_core:,} não compraram chopp:",
    "o recorte de negócio quase não custa nada, porque a carteira já é de chopp.",
    "",
    "O corte por histórico é o degrau caro. Exigir 3+ compras derruba a base a",
    f"{int(((dados['CORE_BUSINESS'] == 1) & (dados['FREQUENCIA_COMPRAS'] > 2)).sum()):,} clientes — a seção 12 mostra por que esse número mudou tanto",
    "clientes — a seção 12 mostra por que esse corte ficou inviável.",
])

## 11. Onde colocar a régua do atraso

> **A pergunta:** qual percentual de compromissos em atraso separa a flutuação
> normal do negócio da inadimplência de fato?

Atraso de poucos dias acontece por motivo operacional — feriado bancário,
fechamento de caixa, fim de semana. Uma régua muito sensível transforma esse
ruído em alarme; uma régua muito frouxa só enxerga o devedor já perdido.

O gráfico varre a régua de 10% a 50% e mostra quanta carteira cai em cada
posição. Não existe resposta estatística para essa escolha: ela é de negócio.
O que a análise oferece é o preço de cada opção.

In [0]:
LIMIARES = [0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]
core = dados[dados["CORE_BUSINESS"] == 1]

linhas = []
for limite in LIMIARES:
    atraso_pag = core["TAXA_ATRASO_PAGAMENTO"] > limite
    atraso_com = core["TAXA_ATRASO_COMODATO"] > limite
    for rotulo, alvo in [("OU", atraso_pag | atraso_com), ("E", atraso_pag & atraso_com)]:
        n_risco = int(alvo.sum())
        linhas.append({
            "limiar": limite, "combinador": rotulo, "em_risco": n_risco,
            "pct_risco": n_risco / len(core) * 100,
            "classe_rara": min(n_risco, len(core) - n_risco),
        })
sensibilidade = pd.DataFrame(linhas)

fig, (esq, dir_) = plt.subplots(1, 2, figsize=(12, 3.9))

for rotulo, cor in [("OU", COR_FINANCEIRO), ("E", COR_COMODATO)]:
    parte = sensibilidade[sensibilidade["combinador"] == rotulo]
    esq.plot(parte["limiar"] * 100, parte["pct_risco"], marker="o",
             color=cor, linewidth=2, label=f'combinador "{rotulo}"')
esq.set_xlabel("régua de atraso (%)")
esq.set_ylabel("% da carteira em risco")
esq.set_title("Quanta carteira cai em cada régua", fontsize=11, fontweight="bold")
esq.set_ylim(0, 100)
esq.legend(fontsize=9)

for rotulo, cor in [("OU", COR_FINANCEIRO), ("E", COR_COMODATO)]:
    parte = sensibilidade[sensibilidade["combinador"] == rotulo]
    dir_.plot(parte["limiar"] * 100, parte["classe_rara"], marker="o",
              color=cor, linewidth=2, label=f'combinador "{rotulo}"')
dir_.axhline(50, color="#b8b7b2", linestyle="--", linewidth=1)
dir_.text(11, 56, "piso prático para validar um modelo", fontsize=8, color=COR_APOIO)
dir_.set_xlabel("régua de atraso (%)")
dir_.set_ylabel("tamanho da classe minoritária")
dir_.set_title("Quanto sobra da classe rara", fontsize=11, fontweight="bold")
dir_.legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f"{'régua':>7}{'OU: em risco':>16}{'classe rara':>13}{'E: em risco':>15}{'classe rara':>13}")
print("-" * 66)
for limite in LIMIARES:
    ou = sensibilidade[(sensibilidade["limiar"] == limite) & (sensibilidade["combinador"] == "OU")].iloc[0]
    e = sensibilidade[(sensibilidade["limiar"] == limite) & (sensibilidade["combinador"] == "E")].iloc[0]
    print(f"{limite:>6.0%}{ou['em_risco']:>10,} ({ou['pct_risco']:>4.1f}%)"
          f"{int(ou['classe_rara']):>13,}{e['em_risco']:>9,} ({e['pct_risco']:>4.1f}%)"
          f"{int(e['classe_rara']):>13,}")

leitura("O que isso significa", [
    "A carteira tem atraso crônico: mesmo com régua em 50%, a maior parte dela",
    "continua classificada como risco pelo combinador OU.",
    "",
    'O combinador escolhe o que é "risco" quando só um dos dois lados atrasa:',
    '  "OU" — atrasou o boleto OU segurou a chopeira. Protege os dois ativos,',
    "         mas deixa a classe boa pequena demais para o modelo aprender.",
    '  "E"  — precisa atrasar nos dois. Classes equilibradas, mas trata quem só',
    "         deve dinheiro como bom pagador.",
    "",
    "Não há escolha tecnicamente correta: é uma decisão de política de crédito.",
    "O notebook 08 mede o que cada uma custa em desempenho de modelo.",
])

## 12. Por que uma taxa sobre poucas compras não significa nada

> **A pergunta:** a partir de quantas compras a taxa de atraso descreve conduta,
> e não acaso?

Uma taxa é uma divisão. Quando o denominador é pequeno, o resultado só pode
assumir alguns valores — e nenhum deles é uma medida de comportamento:

| compras | valores possíveis da taxa |
|:--|:--|
| 1 | 0% ou 100% |
| 2 | 0%, 50% ou 100% |
| 3 | 0%, 33%, 67% ou 100% |
| 5+ | granularidade suficiente para distinguir eventual de habitual |

Um cliente que comprou uma vez e atrasou tem taxa de 100%. O número está certo e
não diz nada: não há reincidência para observar. É **discretização**, não risco.

Isso cria uma tensão real. Exigir histórico torna a métrica confiável e encolhe a
base; aceitar todo mundo preserva a base e admite ruído. A tabela abaixo põe preço
nos dois lados.

In [0]:
core = dados[dados["CORE_BUSINESS"] == 1]

print(f"{'mínimo':>10}{'clientes':>10}{'% base':>9}{'% risco':>10}{'classe rara':>13}{'CV 5 folds':>13}")
print("-" * 65)
for minimo in [0, 1, 2, 3, 5, 10]:
    sub = core[core["FREQUENCIA_COMPRAS"] > minimo]
    if sub.empty:
        continue
    alvo = ((sub["TAXA_ATRASO_PAGAMENTO"] > LIMITE_RISCO)
            | (sub["TAXA_ATRASO_COMODATO"] > LIMITE_RISCO)).astype(int)
    rara = min(int(alvo.sum()), int((alvo == 0).sum()))
    por_fold = rara / 5
    aviso = "  inviável" if por_fold < 5 else ""
    print(f"{'> ' + str(minimo):>10}{len(sub):>10,}{len(sub) / len(core) * 100:>8.0f}%"
          f"{alvo.mean() * 100:>9.0f}%{rara:>13,}{por_fold:>10.1f}/fold{aviso}")

distribuicao = core["FREQUENCIA_COMPRAS"].value_counts().sort_index()
uma_compra = int((core["FREQUENCIA_COMPRAS"] <= 1).sum())

fig, ax = plt.subplots(figsize=(9, 3.4))
corte = distribuicao[distribuicao.index <= 15]
ax.bar(corte.index, corte.values, color=COR_FINANCEIRO, alpha=0.85, width=0.7)
ax.set_xlabel("compras registradas")
ax.set_ylabel("clientes")
ax.set_title("Quantas compras cada cliente tem (core business)", fontsize=11, fontweight="bold")
ax.axvline(1.5, color=COR_COMODATO, linestyle="--", linewidth=1.5)
ax.text(1.8, corte.max() * 0.8, "taxa só pode\nvaler 0% ou 100%",
        fontsize=8.5, color=COR_COMODATO)
plt.tight_layout()
plt.show()

leitura("O que isso significa", [
    f"{uma_compra:,} clientes ({uma_compra / len(core) * 100:.0f}% do core) têm no máximo uma compra.",
    f"A mediana da carteira é {core['FREQUENCIA_COMPRAS'].median():.0f} compra(s).",
    "",
    "FREQUENCIA_COMPRAS conta venda de chopp, não movimentação de comodato:",
    "por isso a mediana é baixa. Com essa contagem, exigir 3+ compras deixa uma",
    "fração da carteira — e uma classe rara pequena demais para validar qualquer",
    "modelo (6 casos, 1,2 por fold).",
    "",
    "Por isso o notebook 04 usa MIN_COMPRAS = 0: não porque o ruído da taxa",
    "sobre poucas compras sumiu, mas porque cortar custa mais do que aceitar.",
    "O notebook 08 mede os dois cenários lado a lado em vez de supor.",
])

## 13. O cliente que alterna forma de pagamento

> **A pergunta:** comprar sempre à vista, sempre a prazo, ou alternar entre os
> dois — qual desses perfis carrega mais risco?

`PCT_COMPRAS_A_PRAZO` é uma variável **pré-fato**: ela é decidida no ato do
pedido, antes de existir qualquer atraso. Isso a torna explicativa de verdade,
diferente das taxas de atraso, que são o desfecho.

A intuição diz que o risco cresce com o prazo. Os dados dizem outra coisa.

In [0]:
core = dados[dados["CORE_BUSINESS"] == 1].copy()
core["EM_RISCO"] = ((core["TAXA_ATRASO_PAGAMENTO"] > LIMITE_RISCO)
                    | (core["TAXA_ATRASO_COMODATO"] > LIMITE_RISCO)).astype(int)

FAIXAS_PRAZO = [-0.01, 0.001, 0.5, 0.99, 1.0]
ROTULOS_PRAZO = ["só à vista", "até 50% a prazo", "51-99% a prazo", "100% a prazo"]
core["MIX"] = pd.cut(core["PCT_COMPRAS_A_PRAZO"], FAIXAS_PRAZO, labels=ROTULOS_PRAZO)

mix = (core.groupby("MIX", observed=True)
       .agg(clientes=("EM_RISCO", "size"), risco=("EM_RISCO", "mean"),
            gasto=("TOTAL_GASTO", "median"))
       .reset_index())

fig, ax = plt.subplots(figsize=(9, 3.6))
barras = ax.bar(mix["MIX"].astype(str), mix["risco"] * 100,
                color=[COR_NEUTRA if r < mix["risco"].max() else COR_COMODATO
                       for r in mix["risco"]], width=0.6)
ax.set_ylabel("% em risco")
ax.set_ylim(0, 105)
ax.set_title("Risco por mix de forma de pagamento", fontsize=11, fontweight="bold")
for barra, (_, linha) in zip(barras, mix.iterrows()):
    ax.text(barra.get_x() + barra.get_width() / 2, linha["risco"] * 100 + 2,
            f"{linha['risco'] * 100:.0f}%\n(n={int(linha['clientes']):,})",
            ha="center", fontsize=8.5)
plt.tight_layout()
plt.show()

print(f"{'mix de pagamento':<20}{'clientes':>10}{'% em risco':>12}{'gasto mediano':>16}")
print("-" * 58)
for _, linha in mix.iterrows():
    print(f"{str(linha['MIX']):<20}{int(linha['clientes']):>10,}"
          f"{linha['risco'] * 100:>11.0f}%{moeda(linha['gasto']):>16}")

so_vista = mix[mix["MIX"] == "só à vista"]["risco"].iloc[0] * 100
alterna = mix[mix["MIX"] == "51-99% a prazo"]["risco"].iloc[0] * 100
todo_prazo = mix[mix["MIX"] == "100% a prazo"]["risco"].iloc[0] * 100

leitura("O que isso significa", [
    f"Quem paga sempre à vista tem o menor risco ({so_vista:.0f}%) — esperado:",
    "sem prazo, não há o que atrasar.",
    "",
    f"Mas quem ALTERNA ({alterna:.0f}%) tem risco maior que quem sempre",
    f"compra a prazo ({todo_prazo:.0f}%). A relação não é monotônica, e é",
    "contraintuitiva: mais prazo não significa mais risco.",
    "",
    "Uma leitura possível é que alternar sinaliza aperto de caixa pontual —",
    "o cliente pede prazo quando não consegue pagar à vista. Isso é hipótese,",
    "não conclusão: a EDA não testa causalidade.",
    "",
    "De qualquer forma, é o tipo de relação que uma régua univariada não captura",
    "e que justifica usar modelos não lineares.",
])

## 14. O que foi deliberadamente deixado de fora

> **A pergunta:** como garantir que o modelo não use identificação pessoal nem
> variáveis que já contenham a resposta?

Três regras de governança atravessam o pipeline inteiro. Elas não são comentário
de documentação: são checagens que **falham a carga** nos notebooks 01 e 04.

**Identificação nominal.** `NM_PESSOA`, `DS_FANTASIA`, CPF, CNPJ, e-mail, telefone
e endereço nunca entram. Nome não prevê inadimplência — o que ele acrescenta é
viés, porque o modelo passa a reconhecer clientes específicos em vez de aprender
conduta. A unidade de análise é `ID_PESSOA` do começo ao fim.

**Causalidade invertida.** `FL_BLOQUEADO` e `DS_MOTIVO_BLOQUEIO` foram auditados
no ERP e descartados: o sistema bloqueia o pedido **porque** o título já venceu, e
o motivo chega a trazer o texto literal *"TITULO ABERTO VENCIDO"*. É o alvo
escrito por extenso dentro de uma coluna que pareceria explicativa.

**Componentes aritméticos do alvo.** `TAXA_ATRASO_*`, `PARCELAS_ATRASADAS`,
`COMODATOS_ATRASADOS`, `RISCO_*` e `AGING_*` constroem o alvo e não podem ser
feature — um modelo treinado com elas apenas recalcula a própria regra.

A exceção reconhecida é `MEDIA_DIAS_ATRASO_PAG/COM`: compartilham origem com o
alvo e ficaram no modelo conscientemente. O notebook 08 mede exatamente quanto
elas valem.

In [0]:
PADROES_NOMINAIS = ("NOME", "NM_", "FANTASIA", "RAZAO", "CPF", "CNPJ",
                    "EMAIL", "TELEFONE", "ENDERECO")
COMPONENTES_DO_ALVO = [
    "TAXA_ATRASO_PAGAMENTO", "TAXA_ATRASO_COMODATO",
    "PARCELAS_ATRASADAS", "COMODATOS_ATRASADOS",
    "RISCO_FINANCEIRO", "RISCO_COMODATO", "PERFIL_RISCO",
    "MAX_DIAS_ATRASO_PAG", "MAX_DIAS_ATRASO_COM",
    "AGING_PAGAMENTO", "AGING_COMODATO",
]
VAZAMENTO_RECONHECIDO = ["MEDIA_DIAS_ATRASO_PAG", "MEDIA_DIAS_ATRASO_COM"]

nominais = [c for c in dados.columns if any(p in c.upper() for p in PADROES_NOMINAIS)]
presentes_alvo = [c for c in COMPONENTES_DO_ALVO if c in dados.columns]

print("=" * 68)
print(f"{'AUDITORIA DE GOVERNANÇA DO CONSOLIDADO':^68}")
print("=" * 68)
print(f"  colunas no consolidado          {dados.shape[1]:>3}")
print(f"  identificação nominal           {len(nominais):>3}  "
      f"{'← ok, nenhuma' if not nominais else nominais}")
print(f"  componentes do alvo presentes   {len(presentes_alvo):>3}  "
      "(existem aqui, proibidos como feature no 04)")
print(f"  vazamento reconhecido           {len(VAZAMENTO_RECONHECIDO):>3}  "
      f"{VAZAMENTO_RECONHECIDO}")

if nominais:
    raise RuntimeError(
        f"Identificação nominal no consolidado: {nominais}. O notebook 01 "
        "deveria ter barrado — verifique o contrato de colunas."
    )

print("\nO consolidado carrega os componentes do alvo de propósito: a EDA precisa")
print("deles para descrever o risco. Quem os proíbe como feature é o notebook 04,")
print("cuja guarda falha se qualquer um aparecer na lista de features.")

leitura("Onde cada regra é aplicada", [
    "notebook 01 — barra identificação nominal na extração (falha a carga).",
    "notebook 04 — barra componentes do alvo como feature (falha a carga).",
    "notebook 08 — mede o custo do vazamento reconhecido, em vez de só declará-lo.",
])

## 15. Resumo

O que a exploração estabeleceu, e o que ela deliberadamente **não** estabelece.

In [0]:
em_risco = int((dados["PERFIL_RISCO"] != "SEM RISCO").sum())
duplo = int((dados["PERFIL_RISCO"] == "RISCO DUPLO").sum())
exposto = float(dados[dados["PERFIL_RISCO"] != "SEM RISCO"]["TOTAL_GASTO"].sum())

print("=" * 66)
print(f"{'RESUMO DA CARTEIRA':^66}")
print("=" * 66)
print(f"  Clientes cadastrados          {total:>10,}")
print(f"  Compraram chopp/chopeira      {int(dados['CORE_BUSINESS'].sum()):>10,}")
print(f"  Em risco (algum tipo)         {em_risco:>10,}  ({em_risco / total * 100:.0f}%)")
print(f"  Em risco duplo                {duplo:>10,}  ({duplo / total * 100:.0f}%)")
print(f"  Faturamento exposto           {moeda(exposto):>10}")
print("=" * 66)

leitura("O que a EDA NÃO responde", [
    "1. Não há corte no tempo: as características e o atraso vêm do mesmo",
    "   período. Isso descreve o que já aconteceu — não prevê o próximo mês.",
    "",
    "2. Os grupos com mais risco são indícios, não causas. Nenhuma diferença",
    "   foi testada estatisticamente.",
    "",
    "3. 'Risco' aqui é a régua de 20% de atraso, uma escolha de negócio.",
    "   Mudar o percentual muda quem é considerado arriscado.",
    "",
    "O notebook 04 transforma essa leitura em alvo de modelo, e os 05 a 07",
    "verificam se o padrão é forte o bastante para ser aprendido.",
])